# Financial Reasoning — LLM Fine-tuning (SFT + GRPO)

**Run order:** Top to bottom, one section at a time.

| Phase | What happens | Time |
|---|---|---|
| 0 · Setup | GPU check, install, auth, upload data | ~10 min |
| 1 · Baseline | Qwen2.5-7B on our val set before any tuning | ~20 min |
| 2 · SFT | Teach model the reasoning format | ~8–12 h |
| 3 · GRPO | Teach model to get answers correct | ~6–10 h |
| 4 · Final eval | Compare all phases, push to HuggingFace | ~20 min |

> **Data was generated locally.** You ran `01_data_pipeline.py` on your Mac.
> Section 0-C below shows exactly how to upload it.

---
## 0-A · Runtime Check

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Go to Runtime → Change runtime type → GPU (A100 recommended)')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU : {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')

# Pick model size based on VRAM
if vram_gb >= 38:   # A100 40/80 GB
    BASE_MODEL = 'Qwen/Qwen2.5-14B-Instruct'
    LOAD_4BIT  = False
    USE_BF16   = True
    print('✅ A100 — will use 14B model (best results)')
else:               # T4 15 GB
    BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
    LOAD_4BIT  = True
    USE_BF16   = False
    print('⚠️  T4 — will use 7B model with 4-bit quantization')

print(f'\nBase model : {BASE_MODEL}')

---
## 0-B · Install Dependencies

In [ ]:
# Unsloth must be installed before everything else
!pip install 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' -q
!pip install trl>=0.12.0 transformers>=4.45.0 datasets>=2.20.0 peft>=0.12.0 -q
!pip install wandb openai pyyaml python-dotenv huggingface_hub tqdm -q
print('\n✅ All packages installed')

---
## 0-C · Upload Your Data

You have two options. **Option A (Google Drive)** is simpler.

### Option A — Google Drive (recommended for beginners)
1. Open [drive.google.com](https://drive.google.com)
2. Create this folder path: `My Drive / llm-ft / processed /`
3. Upload these 3 files from your Mac's `data/processed/` folder:
   - `sft_train_v1.jsonl` (~135 MB)
   - `sft_val_v1.jsonl` (~14 MB)
   - `rl_raw.jsonl` (~7 MB)
4. Run the Drive cell below.

### Option B — HuggingFace Hub (version controlled, reusable)
Run the HF push cell on your **Mac** first, then load it here in one line.
Better for long-term projects because data is versioned and hash-tracked.

In [ ]:
# ── OPTION A: Mount Google Drive and copy data ──────────────────
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount('/content/drive')

# Adjust this path if you put the files in a different Drive folder
DRIVE_DATA = '/content/drive/MyDrive/llm-ft/processed'
LOCAL_DATA = '/content/project/data/processed'

Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)

for fname in ['sft_train_v1.jsonl', 'sft_val_v1.jsonl', 'rl_raw.jsonl']:
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'{LOCAL_DATA}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'✅ {fname} ({size_mb:.0f} MB)')
    else:
        print(f'❌ Not found: {src}')
        print(f'   Upload it to Google Drive at: {DRIVE_DATA}/')

In [ ]:
# ── OPTION B: Load from HuggingFace Hub ─────────────────────────
# Run this on your Mac FIRST to push the data:
#
#   pip install huggingface_hub datasets
#   huggingface-cli login
#   python -c "
#   import json
#   from datasets import Dataset
#   for split, path in [('train','sft_train_v1'),('val','sft_val_v1'),('rl','rl_raw')]:
#       with open(f'data/processed/{path}.jsonl') as f:
#           data = [json.loads(l) for l in f]
#       Dataset.from_list(data).push_to_hub(
#           'YOUR_USERNAME/financial-reasoning-data',
#           split=split, private=True)
#   print('Done')
#   "
#
# Then run THIS cell on Colab:
from datasets import load_dataset
from pathlib import Path
import json

LOCAL_DATA = '/content/project/data/processed'
Path(LOCAL_DATA).mkdir(parents=True, exist_ok=True)

HF_REPO = 'YOUR_USERNAME/financial-reasoning-data'  # ← change this

for split, fname in [('train','sft_train_v1'),('val','sft_val_v1'),('rl','rl_raw')]:
    ds = load_dataset(HF_REPO, split=split)
    out = f'{LOCAL_DATA}/{fname}.jsonl'
    with open(out, 'w') as f:
        for ex in ds:
            f.write(json.dumps(dict(ex)) + '\n')
    print(f'✅ {fname}.jsonl — {len(ds)} examples')

In [ ]:
# ── Validate data before proceeding ─────────────────────────────
import json, re
from pathlib import Path

LOCAL_DATA = '/content/project/data/processed'

for fname in ['sft_train_v1.jsonl', 'sft_val_v1.jsonl', 'rl_raw.jsonl']:
    path = Path(LOCAL_DATA) / fname
    if not path.exists():
        print(f'❌ Missing: {fname} — run Option A or B above')
        continue
    examples = [json.loads(l) for l in open(path)]
    has_think = sum(1 for ex in examples
                    if re.search(r'<think>.*?</think>', ex.get('reasoning_trace',''), re.DOTALL))
    print(f'✅ {fname}: {len(examples)} examples | think tag: {has_think/len(examples):.0%}')

# Show one example so you know what you're training on
print('\n--- Sample training example ---')
ex = json.loads(open(f'{LOCAL_DATA}/sft_train_v1.jsonl').readline())
print('Question:', ex['question'])
print('Answer  :', ex['answer'])
print('Trace   :', ex['reasoning_trace'][:300], '...')

---
## 0-D · Authentication

In [ ]:
import os

# Add these in Colab Secrets (left sidebar → key icon 🔑)
# Names must match exactly: WANDB_API_KEY, HF_TOKEN, OPENROUTER_API_KEY
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY']      = userdata.get('WANDB_API_KEY')
    os.environ['HF_TOKEN']           = userdata.get('HF_TOKEN')
    os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
    print('✅ Loaded secrets from Colab Secrets')
except Exception:
    # Fallback: paste directly (less secure)
    os.environ['WANDB_API_KEY']      = 'paste_here'
    os.environ['HF_TOKEN']           = 'paste_here'
    os.environ['OPENROUTER_API_KEY'] = 'paste_here'
    print('⚠️  Using hardcoded keys — use Colab Secrets instead')

import wandb
wandb.login(key=os.environ['WANDB_API_KEY'])

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('✅ W&B and HuggingFace authenticated')

---
## 0-E · Clone Project Repo

In [ ]:
import os
from pathlib import Path

REPO_URL    = 'https://github.com/YOUR_USERNAME/llm-fine-tunning-financial-reasoning.git'
PROJECT_DIR = '/content/project'

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}

# If data was copied to /content/project/data/processed via Drive,
# skip this. Otherwise ensure the processed folder is in place:
Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('results').mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)

print('Working dir:', os.getcwd())
!ls

---
## 1 · Baseline Evaluation

Run Qwen2.5-7B on the validation set **before any fine-tuning**.  
This score is your comparison point for every subsequent phase.

Expected result: **~35–50% accuracy** (the model can reason but makes many arithmetic errors without domain fine-tuning).

In [ ]:
# Quick sanity check — sample 20 questions to see what the model currently does
import json, random, re
from pathlib import Path

with open('data/processed/sft_val_v1.jsonl') as f:
    val_data = [json.loads(l) for l in f]

print(f'Validation set: {len(val_data)} examples')
print(f'\nFirst 3 questions:')
for ex in val_data[:3]:
    print(f'  Q: {ex["question"]}')
    print(f'  A: {ex["answer"]}')
    print()

In [ ]:
# Run full baseline evaluation
# ~20 min on A100 for full val set (650 examples)
# Use --limit 50 for a quick 3-minute check first

!python scripts/00_setup_eval.py \
    --model {BASE_MODEL} \
    --batch_size 4

# To run a quick check (50 examples, ~3 min) use:
# !python scripts/00_setup_eval.py --model {BASE_MODEL} --limit 50

In [ ]:
# Load and store baseline score for comparison in later phases
import json

with open('results/baseline/baseline_scores.json') as f:
    baseline_data = json.load(f)

BASELINE_ACCURACY = baseline_data['metrics']['accuracy']

print('=== BASELINE RESULTS (no fine-tuning) ===')
print(f'Model    : {baseline_data["model"]}')
print(f'Accuracy : {BASELINE_ACCURACY:.1%}')
print(f'Mean score: {baseline_data["metrics"]["mean_score"]:.3f}')
print(f'Has <think> tag : {baseline_data["metrics"]["format_think_pct"]:.1%}')
print(f'Has <answer> tag: {baseline_data["metrics"]["format_answer_pct"]:.1%}')
print(f'\nTarget after SFT+GRPO: >{BASELINE_ACCURACY + 0.15:.1%} (+15 pp)')

---
## 2 · SFT Training

**Goal:** Teach the model the FORMAT of financial reasoning (`<think>` + `<answer>` tags).  
After SFT, the model should use structured reasoning on every response — even if the numbers aren't always right.

**Expected time:** 8–12h (A100, 14B) | 4–6h (T4, 7B)  
**Expected gain:** +15–20 pp accuracy over baseline

In [ ]:
# Auto-adjust config for this GPU
import yaml, torch

with open('config/sft_v1.yaml') as f:
    sft_cfg = yaml.safe_load(f)

sft_cfg['model']['name']         = BASE_MODEL
sft_cfg['model']['load_in_4bit'] = LOAD_4BIT
sft_cfg['training']['bf16']      = USE_BF16
sft_cfg['training']['fp16']      = not USE_BF16

with open('config/sft_v1.yaml', 'w') as f:
    yaml.dump(sft_cfg, f)

print('SFT config:')
print(f'  model       : {sft_cfg["model"]["name"]}')
print(f'  load_in_4bit: {sft_cfg["model"]["load_in_4bit"]}')
print(f'  bf16        : {sft_cfg["training"]["bf16"]}')
print(f'  lr          : {sft_cfg["training"]["learning_rate"]}')
print(f'  epochs      : {sft_cfg["training"]["epochs"]}')

In [ ]:
!python scripts/02_sft_train.py --config config/sft_v1.yaml

In [ ]:
# Evaluate SFT model on the same val set
!python scripts/00_setup_eval.py \
    --model checkpoints/sft/best \
    --batch_size 4

# Load result
import json
with open('results/baseline/baseline_scores.json') as f:
    sft_result = json.load(f)

SFT_ACCURACY = sft_result['metrics']['accuracy']

print(f'Baseline : {BASELINE_ACCURACY:.1%}')
print(f'After SFT: {SFT_ACCURACY:.1%}  (+{(SFT_ACCURACY-BASELINE_ACCURACY)*100:.1f} pp)')

if SFT_ACCURACY >= BASELINE_ACCURACY + 0.10:
    print('✅ Gate passed — proceed to GRPO')
else:
    print('⚠️  Small gain — check W&B for loss curves before proceeding')

---
## 3 · GRPO Training

**Goal:** Teach the model to get answers **correct** through trial-and-error.  
GRPO generates 8 attempts per question and learns from which ones get the right answer.

**Expected time:** 6–10h (A100)  
**Expected gain:** +3–5 pp over SFT

⚠️ Watch W&B during training — stop if `entropy < 0.15` (collapse signal).

In [ ]:
# Verify SFT checkpoint exists before starting GRPO
from pathlib import Path
assert Path('checkpoints/sft/best').exists(), 'SFT checkpoint missing — complete Section 2 first'
print('✅ SFT checkpoint found')
!ls checkpoints/sft/best/

In [ ]:
!python scripts/03_grpo_train.py --config config/grpo_v1.yaml

In [ ]:
# Evaluate GRPO model
!python scripts/00_setup_eval.py \
    --model checkpoints/grpo/best \
    --batch_size 4

import json
with open('results/baseline/baseline_scores.json') as f:
    grpo_result = json.load(f)

GRPO_ACCURACY = grpo_result['metrics']['accuracy']

print('=== COMPARISON ===')
print(f'Baseline       : {BASELINE_ACCURACY:.1%}')
print(f'After SFT      : {SFT_ACCURACY:.1%}  (+{(SFT_ACCURACY-BASELINE_ACCURACY)*100:.1f} pp)')
print(f'After SFT+GRPO : {GRPO_ACCURACY:.1%}  (+{(GRPO_ACCURACY-BASELINE_ACCURACY)*100:.1f} pp)')

BEST_MODEL = 'checkpoints/grpo/best' if GRPO_ACCURACY >= SFT_ACCURACY else 'checkpoints/sft/best'
print(f'\nBest model: {BEST_MODEL}')

---
## 4 · Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

HF_REPO_ID = 'YOUR_USERNAME/financial-reasoning-qwen'  # ← change this

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=BEST_MODEL, repo_id=HF_REPO_ID, repo_type='model')
print(f'✅ Model at: https://huggingface.co/{HF_REPO_ID}')

---
## Appendix — Useful One-liners

In [ ]:
# Check GPU memory usage at any point
!nvidia-smi --query-gpu=name,memory.used,memory.free --format=csv,noheader

In [ ]:
# Free GPU memory between runs
import torch, gc
try: del model
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f'Available VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

In [ ]:
# Backup checkpoints to Drive (run this periodically to avoid losing work)
import shutil
from google.colab import drive
drive.mount('/content/drive')

BACKUP_DIR = '/content/drive/MyDrive/llm-ft/checkpoints'
shutil.copytree('checkpoints', BACKUP_DIR, dirs_exist_ok=True)
print(f'✅ Checkpoints backed up to {BACKUP_DIR}')